[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoAvancadoIME/blob/main/03_estimadores_robustos.ipynb)


# Ajustamento Básico - Estimadores robustos

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Entender a limitação do MMQ diante de outliers.
2. Apresentar estimadores robustos baseados em funções de perda.
3. Implementar o método IRLS para reponderação iterativa.
4. Comparar MMQ clássico e ajuste robusto em exemplos simples.

**Referências:**

- Huber, P. J., & Ronchetti, E. M. (2009). *Robust Statistics* (2nd ed.). Wiley.
- Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley.


## O Problema

O MMQ minimiza $\sum v_i^2$.

Essa escolha penaliza fortemente resíduos grandes. Por isso, uma única observação contaminada pode puxar a solução ajustada.


## 1. Função de perda

O MMQ clássico usa a função de perda quadrática:

$$
\rho(v)=v^2
$$

Estimadores robustos substituem essa função por perdas que crescem mais lentamente quando o resíduo é grande.


### **1.1 Ideia robusta**

Observações compatíveis recebem peso normal.

Observações muito discrepantes recebem peso reduzido.

O objetivo não é esconder erro grosseiro, mas reduzir seu efeito enquanto ele é investigado.


## 2. Estimador de Huber

A função de Huber combina comportamento quadrático e linear:

$$
\rho(r)=
\begin{cases}
\frac{1}{2}r^2, & |r| \leq c \
c|r| - \frac{1}{2}c^2, & |r| > c
\end{cases}
$$

Resíduos pequenos são tratados como no MMQ; resíduos grandes são amortecidos.


## 3. Reponderação iterativa (IRLS)

O método IRLS resolve uma sequência de MMQs ponderados.

Em cada iteração:

1. ajusta-se o modelo;
2. calculam-se resíduos padronizados;
3. atualizam-se os pesos robustos;
4. repete-se até convergir.


### **3.1 Peso de Huber**

Para resíduo padronizado $r_i$:

$$
w_i =
\begin{cases}
1, & |r_i| \leq c \
\frac{c}{|r_i|}, & |r_i| > c
\end{cases}
$$

O parâmetro $c$ controla a agressividade da robustez.


**Exercício 01 (resolvido):** Compare MMQ clássico e Huber em um ajuste de reta com um outlier.


In [ ]:
import numpy as np

x = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y = np.array([1.0, 2.0, 3.1, 9.0, 5.0, 6.2], dtype=float)
A = np.column_stack([x, np.ones_like(x)])
L = y.reshape(-1, 1)

def wls(A, L, w):
    P = np.diag(w)
    return np.linalg.inv(A.T @ P @ A) @ A.T @ P @ L

X_lsq = wls(A, L, np.ones(len(y)))

w = np.ones(len(y))
c = 1.5
for _ in range(10):
    X = wls(A, L, w)
    v = (A @ X - L).ravel()
    s = 1.4826 * np.median(np.abs(v - np.median(v)))
    s = max(s, 1e-12)
    r = v / s
    w = np.where(np.abs(r) <= c, 1.0, c / np.abs(r))

print("MMQ clássico [m, b]:", X_lsq.ravel())
print("Huber IRLS [m, b]:", X.ravel())
print("Pesos robustos finais:", np.round(w, 3))


## 4. Outros estimadores robustos

**Tukey biweight:** zera a influência de resíduos muito grandes.

**Cauchy:** reduz suavemente a influência de resíduos extremos.

**L1:** minimiza a soma dos módulos dos resíduos e é menos sensível a outliers que o MMQ.


### **4.1 Função de influência**

A função de influência indica quanto uma observação afeta a solução.

No MMQ, a influência cresce sem limite.

Em estimadores robustos, a influência é limitada ou até anulada para resíduos muito grandes.


## 5. Robustez não substitui controle de qualidade

O ajuste robusto é uma ferramenta de mitigação.

A decisão final sobre remover, manter ou reponderar uma observação deve considerar o caderno de campo, a geometria da rede, a qualidade do equipamento, a hipótese física do problema e o efeito sobre os parâmetros.


**Exercício 02 (resolvido):** Observe o efeito do parâmetro $c$ no estimador de Huber.


In [ ]:
cs = [0.8, 1.5, 3.0]
for c in cs:
    w = np.ones(len(y))
    for _ in range(10):
        X = wls(A, L, w)
        v = (A @ X - L).ravel()
        s = 1.4826 * np.median(np.abs(v - np.median(v)))
        s = max(s, 1e-12)
        r = v / s
        w = np.where(np.abs(r) <= c, 1.0, c / np.abs(r))
    print(f"c={c}: X={np.round(X.ravel(), 4)}, pesos={np.round(w, 3)}")


## 6. Aplicações em Geomática

Estimadores robustos aparecem em:

- GNSS urbano com multipercurso;
- fotogrametria com correspondências erradas;
- nuvens de pontos com pontos espúrios;
- ajuste de redes com observações suspeitas;
- integração GNSS/INS com falhas temporárias.


## Lista de exercícios complementares

**Exercício 03:** Insira dois outliers em uma regressão linear e compare MMQ, Huber e Tukey.

**Exercício 04:** Em uma rede de nivelamento, compare remover uma observação suspeita versus aplicar reponderação robusta.

**Exercício 05:** Simule uma trilateração 2D com uma distância contaminada e avalie o deslocamento das coordenadas ajustadas no MMQ clássico e no robusto.
